# LangGraph

### LangGraph란 무엇인가요?

LangGraph는 LLM 애플리케이션에 **"순환(Loop)"**과 "제어(Control)" 기능을 부여하는 라이브러리입니다.

- 기존 LangChain: A → B → C (일방통행, 되돌아가기 어려움)

- LangGraph: A ↔ B (필요하면 다시 돌아감, 상태를 기억함)


이번 실습에서는 LLM이 계산이 필요하면 계산기 도구를 쓰고, 다시 돌아와서 답변하는 '순환 구조'를 만들어 봅니다.

In [ ]:
# LangGraph 맛보기 실습 코드 설명
# 이 코드는 LangGraph의 핵심 개념인 "상태 기반 그래프"와 "도구를 활용한 순환 구조"를 구현한 예제입니다.

# =============================================
# 1단계: 필수 라이브러리 설치
# =============================================
# LangGraph를 사용하기 위해 필요한 패키지들을 설치합니다:
# - langgraph: 그래프 구조로 LLM 워크플로우를 구축하는 라이브러리
# - langchain-openai: OpenAI 모델을 LangChain과 연동
# - langchain-community: 다양한 커뮤니티 도구
# - grandalf: 그래프 시각화를 위한 라이브러리

# 명령어 실행: !는 Colab에서 셀 명령어를 실행하는 특수 기호
!pip install -qU langgraph langchain-openai langchain-community grandalf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.3/157.3 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.3/84.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 1.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:
import os
import getpass

# =============================================
# 2단계: OpenAI API 키 설정
# =============================================
# LLM 모델(gpt-4o-mini)을 사용하기 위해 필요한 API 키를 환경 변수에 저장합니다.
# 보안을 위해 getpass로 키를 입력받아 콘솔에 노출되지 않게 합니다.

# 환경변수에 OPENAI_API_KEY가 없으면 사용자에게 입력 요청
if "OPENAI_API_KEY" not in os.environ:
    # getpass.getpass(): 사용자 입력을 안전하게 받음 (터미널에 표시 안됨)
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key 입력: ")

# =============================================
# 전체 시스템 흐름 개요:
# =============================================
# 1. 사용자 질문이 들어옴 → HumanMessage 생성
# 2. "chatbot" 노드에서 LLM이 질문 분석
#    - 계산이 필요하면 multiply 도구를 호출하도록 요청
#    - 일반 질문이면 바로 답변 생성
# 3. 조건 분기(tools_condition)에서 LLM의 응답 분석:
#    - 도구 호출이 있으면 → "tools" 노드로 이동
#    - 없으면 → 최종 응답으로 종료
# 4. "tools" 노드에서 실제 계산 실행
# 5. 도구 실행 결과를 메시지에 추가하고 다시 "chatbot" 노드로 이동
# 6. LLM이 도구 결과를 받아 최종 답변 생성
# 7. 사용자에게 최종 응답 출력

# =============================================
# 왜 이런 구조를 사용하는가?
# =============================================
# 1. 순환(Loop) 처리: LLM이 한 번에 답변하지 못할 때, 도구를 사용하고 결과를 받아 다시 처리
# 2. 상태 관리: 대화 기록을 중앙에서 관리하면서 노드 간 정보 공유
# 3. 모듈화: 챗봇 로직과 도구 실행 로직을 분리하여 유지보수 용이
# 4. 시각화: 그래프 구조로 전체 워크플로우를 명확히 이해 가능

# 이 코드는 단순 챗봇을 넘어 "도구 사용 → 결과 확인 → 추가 처리" 같은
# 다단계 사고(chain-of-thought) 과정을 구현하는 기본 패턴을 보여줍니다.

OpenAI API Key 입력: ··········


### 상태(State)와 도구(Tool) 정의
LangGraph의 핵심은 **State(상태)**입니다. 이 상태 객체가 노드들 사이를 흘러다니며 대화 내용(messages)을 계속 업데이트합니다.

add_messages: 새로운 대화가 오면 기존 리스트를 덮어쓰지 않고 **추가(append)**하라는 설정입니다.

In [ ]:
# =============================================
# 3단계: 상태(State), 도구(Tool), LLM 설정
# =============================================
# 이 단계에서는 LangGraph의 핵심 개념 3가지를 정의합니다:
# 1. State: 그래프가 실행되는 동안 유지되는 상태 데이터 구조
# 2. Tool: LLM이 사용할 수 있는 외부 기능(여기서는 계산기)
# 3. LLM: 실제 언어 모델과 도구의 바인딩

from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

# =============================================
# 1. 그래프의 상태(State) 정의
# =============================================
# State는 LangGraph의 핵심 개념으로, 그래프의 각 노드가 공유하는 메모리 공간입니다.
# TypedDict를 사용하여 상태의 타입을 명시적으로 정의합니다.
# 이렇게 하면 코드의 가독성이 높아지고 타입 체크가 가능해집니다.

class State(TypedDict):
    # messages: 대화 기록을 저장하는 리스트
    # Annotated: 타입 힌트에 추가 정보를 첨부하는 방법
    # add_messages: LangGraph의 특수한 리듀서(reducer) 함수
    # - 새로운 메시지가 올 때마다 리스트에 추가(append)하도록 지시
    # - 이렇게 하면 각 노드에서 메시지를 수정할 때 전체 리스트를 덮어쓰지 않고 추가만 함
    messages: Annotated[list, add_messages]

# State 객체는 다음과 같이 흘러다닙니다:
# 1. 시작: {"messages": [HumanMessage("123 곱하기 456은 뭐야?")]}
# 2. chatbot 노드 후: {"messages": [HumanMessage(...), AIMessage(도구호출)]}
# 3. tools 노드 후: {"messages": [..., ToolMessage(계산결과)]}
# 4. 다시 chatbot 노드 후: {"messages": [..., AIMessage(최종답변)]}

# =============================================
# 2. 사용할 도구(Tool) 정의
# =============================================
# LLM은 언어 이해는 뛰어나지만 수학 계산에는 약합니다.
# 따라서 계산이 필요할 때 사용할 도구를 만들어 LLM에게 제공합니다.

@tool  # 이 데코레이터는 함수를 LangChain 도구로 변환합니다
def multiply(a: int, b: int) -> int:
    """두 정수의 곱셈 결과를 반환합니다. 계산이 필요할 때 사용하세요."""
    # LLM은 이 함수의 docstring을 읽고 언제 사용해야 하는지 학습합니다
    return a * b

# 도구가 작동하는 방식:
# 1. LLM이 "123 곱하기 456"이라는 질문을 분석
# 2. multiply 도구를 사용해야겠다고 판단
# 3. multiply(123, 456)을 호출하는 구조화된 요청 생성
# 4. 도구 실행 결과를 받아서 사용자에게 설명

# =============================================
# 3. LLM과 도구 연결 (Bind)
# =============================================
# ChatOpenAI: OpenAI의 언어 모델을 사용하기 위한 래퍼 클래스
# model="gpt-4o-mini": 상대적으로 가볍고 저렴한 모델 선택
llm = ChatOpenAI(model="gpt-4o-mini")

# 사용 가능한 도구 리스트 생성 (지금은 multiply 하나만)
tools = [multiply]

# LLM에 도구를 바인딩: 모델이 이 도구들을 인식하고 필요할 때 호출할 수 있게 됨
# bind_tools() 메서드는:
# 1. 각 도구의 설명을 모델에 제공
# 2. 모델이 도구를 호출할 때 필요한 출력 형식을 설정
# 3. 모델이 도구 사용을 학습하도록 프롬프트에 지침 추가
llm_with_tools = llm.bind_tools(tools)

# 이제 llm_with_tools는 다음과 같은 기능을 가집니다:
# - 일반적인 대화 가능
# - 도구 사용이 필요한 상황을 인지하면 도구 호출 요청 생성
# - 도구 호출 시 정해진 형식(JSON)으로 응답

print("상태 정의 및 도구 연결 완료!")
# 이 시점에서 우리는 다음을 준비했습니다:
# 1. 데이터가 흐르는 통로(State)
# 2. LLM이 사용할 도구(multiply)
# 3. 도구 사용법을 아는 LLM(llm_with_tools)

# =============================================
# 다음 단계 예고:
# =============================================
# 다음 코드에서는 이 구성요소들을 연결하여 그래프를 만듭니다:
# - 노드(Node): chatbot(LLM), tools(도구실행)
# - 엣지(Edge): 노드 간 연결 경로
# - 조건부 분기: LLM의 응답에 따라 다른 노드로 이동
# 이 모든 것이 State 객체를 통해 메시지를 주고받으며 작동합니다.

상태 정의 및 도구 연결 완료!


### 노드(Node)와 그래프(Graph) 구성

이제 작업자(Node)를 배치하고 작업 순서(Edge)를 연결합니다.

- Chatbot Node: LLM이 생각하고 답변하는 곳
- Tools Node: LLM이 도구 사용을 요청하면 실제로 실행하는 곳
- Conditional Edge (분기): LLM의 응답을 보고 "도구 쓸래?" 아니면 "답변 하고 끝낼래?"를 결정
- Loop (순환): 도구를 썼으면 반드시 다시 챗봇에게 돌아가서 결과를 보고하도록 연결 (tools -> chatbot)

In [ ]:
# =============================================
# 4단계: 그래프(Graph) 구성 - 노드, 엣지, 조건부 흐름
# =============================================
# 이 단계에서는 개별 컴포넌트(LLM, 도구)를 연결하여 실행 가능한 워크플로우를 만듭니다.
# LangGraph의 핵심은 "그래프" 개념으로, 노드(작업 단위)와 엣지(흐름)로 구성됩니다.

from langgraph.prebuilt import ToolNode, tools_condition

# =============================================
# 1. 챗봇 노드 함수 정의
# =============================================
# 노드(Node): 그래프에서 하나의 작업 단위를 수행하는 함수
# chatbot 노드는 LLM이 사용자 입력을 분석하고 응답을 생성하는 역할을 합니다.

def chatbot(state: State):
    # state["messages"]: 현재까지 누적된 모든 대화 메시지
    # 예: [HumanMessage("123*456?"), ...]

    # llm_with_tools.invoke(): LLM을 호출하여 다음 메시지 생성
    # - 도구가 필요하면 ToolCall 메시지 생성
    # - 필요없으면 일반 AIMessage 생성
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

# 이 함수의 실행 흐름:
# 1. 입력: {"messages": [사용자질문]}
# 2. 처리: LLM이 질문 분석 → 도구 필요성 판단
# 3. 출력: {"messages": [LLM응답]} (State 객체에 추가)

# =============================================
# 2. 그래프 빌더 생성
# =============================================
# StateGraph: LangGraph의 핵심 클래스로, 상태 기반 그래프를 구축합니다
# State: 앞서 정의한 TypedDict - 그래프가 사용할 데이터 구조 지정
graph_builder = StateGraph(State)

# 그래프 빌더는 건축가처럼 노드와 연결을 설계합니다:
# - 노드 추가: add_node()
# - 시작점 설정: set_entry_point()
# - 연결 설정: add_edge(), add_conditional_edges()
# - 최종 빌드: compile()

# =============================================
# 3. 노드 추가 (작업자 배치)
# =============================================
# 그래프에 두 개의 노드를 추가합니다:
# 1. "chatbot": LLM이 생각하고 답변하는 노드
# 2. "tools": 실제 도구를 실행하는 노드

graph_builder.add_node("chatbot", chatbot)  # LLM 처리 담당
graph_builder.add_node("tools", ToolNode(tools))  # 도구 실행 담당

# ToolNode는 LangGraph가 제공하는 전용 노드입니다:
# - 내부 동작: LLM이 요청한 도구를 찾아 실행
# - 입력: {"messages": [AIMessage(tool_calls=[...])]}
# - 출력: {"messages": [ToolMessage(실행결과)]}

# =============================================
# 4. 엣지 연결 (작업 순서 연결)
# =============================================
# 엣지(Edge): 노드 간의 이동 경로를 정의합니다.

# 4.1 시작점 설정
# 그래프 실행이 시작되는 첫 번째 노드를 지정합니다
graph_builder.set_entry_point("chatbot")
# → 모든 실행은 "chatbot" 노드에서 시작

# 4.2 조건부 분기 엣지 추가 (가장 중요한 부분!)
# chatbot 노드 실행 후, LLM의 응답에 따라 다음 경로를 결정합니다
graph_builder.add_conditional_edges(
    "chatbot",      # 출발 노드
    tools_condition, # 조건 판단 함수
)

# tools_condition 함수의 작동 원리:
# 1. LLM 응답(AIMessage)을 검사
# 2. tool_calls 속성이 있으면 → "tools" 노드로 이동
# 3. 없으면 → "__end__" (그래프 종료)로 이동
# 이 분기는 그래프가 "도구를 쓸지 말지"를 동적으로 결정하게 합니다

# 4.3 순환 엣지 추가 (도구 실행 후 다시 챗봇으로)
graph_builder.add_edge("tools", "chatbot")

# 이 엣지가 LangGraph의 핵심 가치인 "순환"을 구현합니다:
# 도구 실행(tools) → 결과 수신 → 다시 LLM 분석(chatbot)
# 이렇게 하면 LLM이 도구 결과를 바탕으로 추가 생각을 할 수 있습니다

# =============================================
# 5. 그래프 컴파일 (실행 가능한 앱으로 변환)
# =============================================
# 모든 노드와 엣지 정의가 끝나면 그래프를 컴파일합니다
graph = graph_builder.compile()

# 컴파일 과정에서:
# 1. 모든 노드와 엣지의 유효성 검사
# 2. 실행 엔진 최적화
# 3. State 객체의 흐름 관리 로직 생성
# 4. 컴파일된 그래프는 call() 또는 stream() 메서드로 실행 가능

# =============================================
# 6. 그래프 구조 시각화
# =============================================
print("그래프 생성 완료! 구조 확인:")
graph.get_graph().print_ascii()

# ASCII 아트 그래프 해석:
#        +-----------+
#        | __start__ |         # 시작점 (가상의 노드)
#        +-----------+
#               *
#               *
#               *
#          +---------+
#          | chatbot |         # 1. LLM 노드 (항상 먼저 실행)
#          +---------+
#          .         .
#        ..           ..
#       .               .
# +---------+         +-------+
# | __end__ |         | tools | # 조건부 분기: 도구 없으면 종료, 있으면 도구 실행
# +---------+         +-------+
#                          *
#                          *
#                          *
#                     +---------+
#                     | chatbot | # 도구 실행 후 다시 LLM 노드로 돌아옴 (순환)
#                     +---------+

# 그래프 실행 시나리오:
# 시나리오 1: 일반 질문
# __start__ → chatbot → (도구 없음) → __end__ [종료]

# 시나리오 2: 계산 질문 (순환 발생)
# __start__ → chatbot → (도구 필요) → tools → chatbot → (도구 결과 처리) → __end__

# 이 구조의 장점:
# 1. 유연성: LLM이 도구 사용 여부를 상황에 따라 결정
# 2. 확장성: 새로운 도구나 노드를 쉽게 추가 가능
# 3. 투명성: 실행 흐름을 시각적으로 이해 가능
# 4. 디버깅: 각 노드의 입출력을 명확히 추적 가능

# =============================================
# 다음 단계 예고:
# =============================================
# 컴파일된 graph 객체는 이제 실행 준비가 되었습니다.
# 다음 코드에서는 이 그래프에 실제 질문을 넣어보고, 각 단계별로 어떤 일이 발생하는지 살펴봅니다.
# stream() 메서드를 사용하면 그래프 실행 과정을 단계별로 관찰할 수 있습니다.

그래프 생성 완료! 구조 확인:
        +-----------+         
        | __start__ |         
        +-----------+         
               *              
               *              
               *              
          +---------+         
          | chatbot |         
          +---------+         
          .         .         
        ..           ..       
       .               .      
+---------+         +-------+ 
| __end__ |         | tools | 
+---------+         +-------+ 


### 실전 테스트
이제 만든 챗봇을 테스트해 봅니다.

- Case 1: 도구가 필요 없는 일상 대화
- Case 2: 도구가 필요한 계산 요청 (순환 발생)

실행 결과를 보면 Case 2에서 Chatbot -> Tools -> Chatbot 순서로 이동하는 것을 볼 수 있습니다.

In [ ]:
# =============================================
# 5단계: 그래프 실행 및 실전 테스트
# =============================================
# 이 단계에서는 구성한 그래프에 실제 질문을 입력하고 실행 과정을 살펴봅니다.
# 그래프가 단계별로 어떻게 동작하는지 관찰할 수 있습니다.

from langchain_core.messages import HumanMessage

# =============================================
# 그래프 실행 함수 정의
# =============================================
# run_chat 함수: 사용자 질문을 받아 그래프를 실행하고 각 단계를 출력합니다
def run_chat(question):
    print(f"\n 질문: {question}")
    print("-" * 40)  # 구분선으로 가독성 향상

    # graph.stream() 메서드: 그래프 실행 과정을 실시간으로 스트리밍
    # 입력: 초기 상태 객체 (HumanMessage 포함)
    # 출력: 각 노드 실행 시점마다 이벤트 발생

    # graph.stream()의 작동 원리:
    # 1. {"messages": [HumanMessage(...)]} 상태로 시작
    # 2. set_entry_point("chatbot")에 따라 chatbot 노드 실행
    # 3. chatbot → 조건부 분기 → tools 또는 __end__
    # 4. tools 실행 후 다시 chatbot으로 돌아감 (순환)
    # 5. 각 노드 실행 시마다 event 객체 생성

    for event in graph.stream({"messages": [HumanMessage(content=question)]}):
        # event 구조: {'노드이름': {'messages': [메시지리스트]}}
        # 예: {'chatbot': {'messages': [HumanMessage, AIMessage]}}

        # 이벤트 딕셔너리의 각 항목 처리 (보통 하나의 key-value 쌍)
        for key, value in event.items():
            print(f" [현재 위치: {key}]")

            # =============================================
            # 챗봇 노드에서 발생한 이벤트 처리
            # =============================================
            if key == "chatbot":
                # value['messages'][-1]: 가장 최근에 추가된 메시지
                msg = value['messages'][-1]

                # 도구 호출이 포함된 응답인지 확인
                if msg.tool_calls:
                    # 도구 호출이 있다면: LLM이 계산기를 사용하려 함
                    print(f"   도구 호출 감지! (이름: {msg.tool_calls[0]['name']}, 값: {msg.tool_calls[0]['args']})")

                    # msg.tool_calls 구조:
                    # [{
                    #   'name': 'multiply',  # 호출할 도구 이름
                    #   'args': {'a': 123, 'b': 456},  # 전달할 인자
                    #   'id': 'call_...'  # 호출 ID
                    # }]

                else:
                    # 도구 호출이 없다면: 일반 답변 또는 최종 답변
                    print(f"   최종 답변: {msg.content}")

            # =============================================
            # 도구 노드에서 발생한 이벤트 처리
            # =============================================
            elif key == "tools":
                # value['messages'][-1]: 도구 실행 결과 메시지
                msg = value['messages'][-1]
                print(f"   도구 실행 결과: {msg.content}")

                # ToolMessage의 content에는 도구 실행 결과가 문자열로 저장됨
                # 예: "56088" (123 * 456의 결과)

# =============================================
# 실행 흐름 시각화 (두 가지 테스트 케이스)
# =============================================
"""
실행 흐름 시나리오 1: 일반 대화
----------------------------------------
1. 초기화: {"messages": [HumanMessage("안녕? 너는 어떤 도구를 쓸 수 있어?")]}
2. chatbot 노드 실행:
   - LLM이 질문 분석: "도구 설명 요청"
   - 도구 호출 필요 없음 → 일반 응답 생성
   - event 출력: [현재 위치: chatbot] 최종 답변: "..."
3. 조건부 분기: tool_calls 없음 → __end__로 이동 (종료)
4. 실행 완료

실행 흐름 시나리오 2: 계산 요청 (순환 발생)
----------------------------------------
1. 초기화: {"messages": [HumanMessage("123 곱하기 456은 뭐야?")]}
2. chatbot 노드 실행 (첫 번째):
   - LLM이 질문 분석: "계산 필요"
   - multiply 도구 호출 결정
   - event 출력: [현재 위치: chatbot] 도구 호출 감지! (이름: multiply, 값: {'a': 123, 'b': 456})
3. 조건부 분기: tool_calls 있음 → tools 노드로 이동
4. tools 노드 실행:
   - multiply(123, 456) 함수 호출
   - 결과 56088 생성
   - ToolMessage 생성
   - event 출력: [현재 위치: tools] 도구 실행 결과: 56088
5. 순환 엣지: tools → chatbot (도구 실행 후 다시 LLM에게 돌아감)
6. chatbot 노드 실행 (두 번째):
   - 입력: 이전 모든 메시지 + 도구 결과
   - LLM이 도구 결과를 바탕으로 최종 답변 구성
   - event 출력: [현재 위치: chatbot] 최종 답변: "123 곱하기 456은 56088입니다."
7. 조건부 분기: tool_calls 없음 → __end__로 이동 (종료)
8. 실행 완료
"""

# =============================================
# 이 코드가 보여주는 LangGraph의 핵심 가치:
# =============================================
# 1. 가시성: 실행 과정을 단계별로 관찰 가능
# 2. 상태 관리: 메시지 히스토리가 자동으로 누적되고 전달됨
# 3. 순환 처리: 도구 사용 → 결과 확인 → 추가 처리 가능
# 4. 유연성: 조건부 분기를 통해 상황에 맞는 경로 선택

# =============================================
# 실제 테스트 실행 (다음 코드에서 실행됨)
# =============================================
"""
# --- Case 1 테스트 실행 ---
run_chat("안녕? 너는 어떤 도구를 쓸 수 있어?")

# --- Case 2 테스트 실행 ---
run_chat("123 곱하기 456은 뭐야?")

실행 결과 예상:
Case 1: chatbot → __end__ (도구 호출 없음)
Case 2: chatbot → tools → chatbot → __end__ (순환 발생)
"""

# =============================================
# 디버깅 및 모니터링 용도:
# =============================================
# 이 코드는 학습용으로 각 단계를 출력하지만, 실제 프로덕션에서는:
# 1. 로깅 레벨 조정 (INFO, DEBUG 등)
# 2. 모니터링 대시보드 연동
# 3. 실행 메트릭 수집 (각 노드 실행 시간, 성공률 등)
# 4. 오류 처리 및 재시도 로직 추가

# =============================================
# 확장 가능성:
# =============================================
# 이 구조는 다양한 방식으로 확장 가능:
# 1. 더 많은 도구 추가 (검색, 데이터베이스 조회, API 호출 등)
# 2. 복잡한 분기 로직 추가 (다중 선택 분기)
# 3. 병렬 실행 노드 추가 (여러 도구 동시 실행)
# 4. 인간 개입 노드 추가 (중요 결정 시 사용자 확인)

# 이 코드는 LangGraph의 가장 기본적인 패턴을 보여줍니다.
# 실제 애플리케이션에서는 이 패턴을 기반으로 더 복잡하고 강력한 워크플로우를 구축할 수 있습니다.

In [ ]:
# =============================================
# 6단계: 실제 테스트 실행 및 결과 분석
# =============================================
# 이 부분에서는 앞서 구성한 LangGraph 시스템에 실제 테스트를 진행합니다.
# 두 가지 다른 유형의 질문을 통해 그래프가 어떻게 다른 경로로 실행되는지 확인합니다.

# --- Case 1 테스트 실행 ---
# 일반적인 대화 질문: 도구 사용 없이 바로 답변 가능한 경우
run_chat("안녕? 너는 어떤 도구를 쓸 수 있어?")

# 실행 결과:
# 질문: 안녕? 너는 어떤 도구를 쓸 수 있어?
# ----------------------------------------
# [현재 위치: chatbot]
#    최종 답변: 안녕하세요! 저는 두 정수의 곱셈 결과를 계산할 수 있는 도구를 사용할 수 있습니다. 필요한 계산이 있다면 말씀해 주세요!

# =============================================
# Case 1 실행 과정 분석:
# =============================================
# 1. 입력: HumanMessage("안녕? 너는 어떤 도구를 쓸 수 있어?")
# 2. chatbot 노드 실행:
#    - LLM 분석: "이 질문은 도구 사용이 필요없는 일반적인 대화 질문"
#    - 도구 호출 없이 바로 답변 생성
#    - 결과: AIMessage("안녕하세요! 저는...")
# 3. 조건부 분기: tool_calls 속성이 없으므로 __end__로 이동
# 4. 그래프 종료
#
# 실행 경로: __start__ → chatbot → __end__
# 순환 발생: NO (단일 패스)

# --- Case 2 테스트 실행 ---
# 계산 요청 질문: 도구 사용이 필요한 경우 (순환 발생)
run_chat("123 곱하기 456은 뭐야?")

# 실행 결과:
# 질문: 123 곱하기 456은 뭐야?
# ----------------------------------------
# [현재 위치: chatbot]
#    도구 호출 감지! (이름: multiply, 값: {'a': 123, 'b': 456})
# [현재 위치: tools]
#    도구 실행 결과: 56088
# [현재 위치: chatbot]
#    최종 답변: 123 곱하기 456은 56,088입니다.


 질문: 안녕? 너는 어떤 도구를 쓸 수 있어?
----------------------------------------
 [현재 위치: chatbot]
   최종 답변: 안녕하세요! 저는 두 정수의 곱셈 결과를 계산할 수 있는 도구를 사용할 수 있습니다. 필요한 계산이 있다면 말씀해 주세요!


In [ ]:
# =============================================
# Case 2 실행 과정 분석:
# =============================================
# 1. 입력: HumanMessage("123 곱하기 456은 뭐야?")
# 2. chatbot 노드 실행 (첫 번째):
#    - LLM 분석: "이 질문은 수학 계산이 필요함"
#    - multiply 도구 호출 결정
#    - 결과: AIMessage(tool_calls=[{'name': 'multiply', 'args': {'a': 123, 'b': 456}}])
# 3. 조건부 분기: tool_calls 속성이 있으므로 tools 노드로 이동
# 4. tools 노드 실행:
#    - multiply(123, 456) 함수 호출
#    - 결과: 56088
#    - ToolMessage("56088") 생성
# 5. 순환 엣지: tools → chatbot (도구 결과를 가지고 다시 LLM에게)
# 6. chatbot 노드 실행 (두 번째):
#    - 입력: [HumanMessage(질문), AIMessage(도구호출), ToolMessage(56088)]
#    - LLM 분석: "도구 결과를 받았으니 최종 답변 생성"
#    - 결과: AIMessage("123 곱하기 456은 56,088입니다.")
# 7. 조건부 분기: tool_calls 속성이 없으므로 __end__로 이동
# 8. 그래프 종료
#
# 실행 경로: __start__ → chatbot → tools → chatbot → __end__
# 순환 발생: YES (총 2번의 chatbot 노드 실행)

# =============================================
# 두 테스트의 비교 분석:
# =============================================
# | 구분         | Case 1 (일반 질문)          | Case 2 (계산 질문)              |
# |--------------|---------------------------|-------------------------------|
# | 실행 경로    | chatbot → __end__         | chatbot → tools → chatbot → __end__ |
# | chatbot 실행 | 1회                       | 2회 (순환 발생)               |
# | 도구 사용    | 없음                      | multiply 도구 사용            |
# | LLM 역할     | 직접 답변 생성             | 1) 도구 호출 결정 2) 결과 해석 |
# | 상태 변화    | messages: [질문, 답변]    | messages: [질문, 도구호출, 결과, 최종답변] |

# =============================================
# LangGraph 시스템의 핵심 작동 원리:
# =============================================
# 1. 상태 기반 실행(State-driven execution):
#    - 모든 노드는 State 객체를 입력받고 수정된 State를 반환
#    - State의 messages 필드는 실행 내역의 완전한 기록

# 2. 조건부 라우팅(Conditional routing):
#    - tools_condition 함수가 LLM 응답을 분석하여 다음 노드 결정
#    - 동적 의사결정: 같은 chatbot 노드라도 상황에 따라 다른 경로

# 3. 순환 구조(Cyclic graph):
#    - 일반적인 LangChain은 직선형(linear) 파이프라인
#    - LangGraph는 도구 실행 후 다시 LLM으로 돌아가는 순환 가능
#    - 이는 인간의 사고 과정(생각→행동→재고)을 모방

# 4. 도구 사용 패턴(Tool usage pattern):
#    - LLM이 도구 필요성 자체를 판단
#    - 도구 호출은 구조화된 형식(ToolCall)으로 표준화
#    - 도구 실행 결과는 ToolMessage로 명확히 표시

# =============================================
# 이 시스템의 실제 적용 가능성:
# =============================================
# 1. 고급 계산기: 더 많은 수학 함수 추가(나눗셈, 제곱근, 통계 등)
# 2. 데이터 조회 시스템: DB 조회 도구 + 결과 분석 도구
# 3. 웹 검색 에이전트: 검색 도구 → 결과 수집 → 요약 도구
# 4. 멀티스텝 작업: 여러 도구를 순차적으로 호출하는 복잡한 작업

# =============================================
# 확장 예시 (향후 발전 방향):
# =============================================
# 1. 다중 도구 지원:
    '''
    @tool
    def divide(a: int, b: int): ...
    @tool
    def search_web(query: str): ...
    @tool
    def get_weather(city: str): ...
    '''

# 2. 인간 개입(Human-in-the-loop):
#    - 특정 조건에서 사용자 확인을 요청하는 노드 추가
#    - 예: 비용이 큰 작업 전 사용자 승인 요청

# 3. 병렬 실행:
#    - 여러 도구를 동시에 실행하는 병렬 노드 추가
#    - 예: 날씨와 뉴스를 동시에 조회

# 4. 에러 처리:
#    - 도구 실행 실패 시 대체 경로 설정
#    - 재시도 로직 구현

# =============================================
# 이 코드가 보여주는 LangGraph의 가치:
# =============================================
# 1. 복잡한 LLM 워크플로우를 명확한 그래프로 시각화
# 2. 상태 관리로 인한 실행 컨텍스트 유지
# 3. 조건부 분기와 순환으로 유연한 흐름 제어
# 4. 도구 사용과 LLM 사고의 자연스러운 통합

# 이로써 우리는 LangGraph의 기본 개념을 맛보았습니다.
# 간단한 계산기에서 시작해 점점 더 복잡한 에이전트 시스템으로 발전시킬 수 있는
# 강력한 기반을 마련했습니다.


 질문: 123 곱하기 456은 뭐야?
----------------------------------------
 [현재 위치: chatbot]
   도구 호출 감지! (이름: multiply, 값: {'a': 123, 'b': 456})
 [현재 위치: tools]
   도구 실행 결과: 56088
 [현재 위치: chatbot]
   최종 답변: 123 곱하기 456은 56,088입니다.
